# 10c Classify Abstract Concepts

Classify each unique abstract once against the full reviewed catalog. The prompt contains only title, abstract, and concept definitions; it receives no SI values, pair split, or forecast outcomes. Results resume from a versioned parquet cache.

The default is a small pilot. Set `MAX_CALLS=None` and enable `RUN_LLM_CLASSIFICATION` only after reviewing notebook 10a's cost estimate and the prompt output.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import cfg, cfg_path, load_pipeline_config
from amr.pipelines.concepts import build_si_concept_paths, load_concept_corpus, corpus_preflight
from amr.si.concepts import (
    SEED_CONCEPTS, concept_catalog_hash, estimate_classification_cost,
    classify_concept_abstracts, build_yearly_concept_signals,
    rolling_origin_concept_forecasts, summarise_incremental_forecast_gains,
    propose_concept_children,
)

PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
CONCEPT_PATHS = build_si_concept_paths(PROJECT_ROOT)
MODEL_NAME = str(cfg(PIPELINE_CONFIG, "llm.model", "gpt-5-nano"))

corpus = pd.read_parquet(CONCEPT_PATHS["corpus"])
abstract_table = pd.read_parquet(CONCEPT_PATHS["abstracts"])
catalog = json.loads(CONCEPT_PATHS["catalog"].read_text())
CATALOG_VERSION = concept_catalog_hash(catalog)
MODEL_NAME = str(cfg(PIPELINE_CONFIG, "llm.model", "gpt-5-nano"))
RUN_LLM_CLASSIFICATION = False
MAX_CALLS = 25  # pilot only; use None for the full unique-abstract corpus
abstracts = abstract_table.loc[abstract_table["abstract_available"], ["pmid", "title", "abstract"]].drop_duplicates("pmid")
print(f"Catalog {CATALOG_VERSION}; {len(abstracts):,} unique abstracts; {len(catalog)} concepts")


## Preview prompt without making a request

In [ ]:
from amr.si.concepts import build_classification_prompt, CONCEPT_SYSTEM_PROMPT
if len(abstracts):
    preview = build_classification_prompt(abstracts.iloc[0]["title"], abstracts.iloc[0]["abstract"], catalog)
    print(CONCEPT_SYSTEM_PROMPT)
    print(preview[:5000])


## Optional resumable classification

In [ ]:
if RUN_LLM_CLASSIFICATION:
    from amr.llm import OpenAIConfig, OpenAILanguageModel
    llm = OpenAILanguageModel(OpenAIConfig(model=MODEL_NAME, temperature=0, max_tokens=1200))
    scores = classify_concept_abstracts(
        abstracts, catalog, llm=llm, model_name=MODEL_NAME,
        cache_path=CONCEPT_PATHS["scores"], max_calls=MAX_CALLS,
    )
    print(scores.groupby(["catalog_version", "status"]).size())
else:
    if CONCEPT_PATHS["scores"].exists():
        scores = pd.read_parquet(CONCEPT_PATHS["scores"])
        print(scores.groupby(["catalog_version", "status"]).size())
    else:
        print("Classification is disabled; no model calls were made.")
